# 01 - Production Data Pipeline

**Complete 5-stage data pipeline with stealth scraping:**

1. 🕵️ **Stealth Scraping** - undetected_chromedriver + rotating proxies + jitter
2. ⚡ **Parallel + Retry** - Tenacity exponential backoff
3. 🧹 **Data Cleaning** - Duplicate detection, validation (Pandas)
4. 🖼️ **Preprocessing** - Resize 224×224, Gaussian filter
5. 💾 **Parquet Storage** - Efficient metadata storage

**Target:** 1000 images/class × 10 classes = 10,000 images

## Setup

In [ ]:
import sys
from pathlib import Path
import os

# Add src to path
sys.path.insert(0, str(Path.cwd().parent))

from src.crawler_stealth import crawl_all_stealth, DATASET_CLASSES
from src.data_cleaning import scan_and_clean_dataset, get_class_distribution, get_image_size_stats
from src.preprocessing import preprocess_dataset
from src.storage import save_metadata_to_parquet, create_dataset_manifest, get_dataset_stats

print('Pipeline modules loaded successfully!')
print(f'Available classes: {len(DATASET_CLASSES)}')

## Stage 1 & 2: Stealth Scraping + Parallel Retry

In [ ]:
# Configuration
MAX_IMAGES_PER_CLASS = 100  # Change to 1000 for production
OUTPUT_ROOT = '..\\data\\raw'
TARGET_CLASSES = list(DATASET_CLASSES.keys())  # or ['Rice_Healthy', 'Rice_Blast', ...]

print(f'Configuration:')
print(f'  Max images/class: {MAX_IMAGES_PER_CLASS}')
print(f'  Output: {OUTPUT_ROOT}')
print(f'  Classes to crawl: {len(TARGET_CLASSES)}')

In [ ]:
# Stage 1 & 2: Stealth crawl with exponential backoff retry
print('Starting stealth crawl with retry logic...')
print('='*70)
print('Features: undetected_chromedriver + random delays + exponential backoff')
print('='*70)

crawl_results = crawl_all_stealth(
    output_root=OUTPUT_ROOT,
    max_images_per_class=MAX_IMAGES_PER_CLASS,
    classes=TARGET_CLASSES[:1]  # Start with 1 class for testing
)

print('\nCrawl Results:')
for class_name, result in crawl_results.items():
    print(f"  {class_name}: {result['downloaded']}/{MAX_IMAGES_PER_CLASS}")

## Stage 3: Data Cleaning (Duplicate Detection + Validation)

In [ ]:
# Stage 3: Clean dataset
print('Starting data cleaning...')
print('='*70)
print('Operations: validate image files, detect duplicates, remove broken images')
print('='*70)

metadata_csv = '..\\data\\raw\\_metadata.csv'
df_clean = scan_and_clean_dataset(
    data_root=OUTPUT_ROOT,
    output_metadata=metadata_csv
)

print('\nClass Distribution:')
print(get_class_distribution(df_clean))

print('\nImage Size Stats:')
print(get_image_size_stats(df_clean))

## Stage 4: Image Preprocessing

In [ ]:
# Stage 4: Preprocess images
print('Starting image preprocessing...')
print('='*70)
print('Operations: resize to 224×224, Gaussian filter, normalization')
print('='*70)

preprocess_stats = preprocess_dataset(
    data_root=OUTPUT_ROOT,
    output_root='..\\data\\processed',
    target_size=(224, 224),
    apply_filter=True,
    normalize=True
)

## Stage 5: Parquet Storage + Manifest

In [ ]:
# Stage 5: Save to Parquet (more efficient than CSV)
import json

metadata_parquet = '..\\data\\raw\\_metadata.parquet'

# Convert CSV to Parquet
save_metadata_to_parquet(df_clean, metadata_parquet)

# Create dataset manifest
manifest = create_dataset_manifest(
    metadata_parquet,
    '..\\data\\.manifest'
)

print('\nDataset Manifest:')
print(json.dumps(manifest, indent=2))

## Pipeline Summary

In [ ]:
# Get final statistics
stats = get_dataset_stats(metadata_parquet)

print('\n' + '='*70)
print('PIPELINE COMPLETE')
print('='*70)
print(json.dumps(stats, indent=2))
print('\nOutput Structure:')
print('  data/')
print('    raw/                    (original crawled images)')
print('    processed/              (resized 224×224 + filtered)')
print('    _metadata.parquet       (efficient metadata storage)')
print('    _metadata.csv           (backup CSV)')
print('  .manifest                 (dataset manifest)')